# Key Outputs:

    effect_comparison: fold change table across interventions

    summary_stats: which intervention most restores youth-like gene expression

    sn_obj@meta.data$Reversal1: per-cell signature scores

    cellchat@net: cell–cell communication network highlighting reversal-associated interactions
    
    conforms to standard multi-omic integration practices, offering both functional and cellular context analyses.
    https://www.r-bloggers.com/2024/05/a-guide-to-designs-and-contrasts-in-deseq2/
    https://nf-co.re/rnaseq/3.19.0/docs/usage/differential_expression_analysis/de_rstudio
    https://hbctraining.github.io/DGE_workshop/lessons/04_DGE_DESeq2_analysis.html

# Installations

# Packages

In [ ]:
# Point reticulate at the Python env that has anndata/scanpy — MUST be set before any Python call.
# If anndata still is not found, change this to the env where `python -c "import anndata"` works.
Sys.setenv(RETICULATE_PYTHON = "/data/bonn-epyc/projects/manuela/conda/envs/mpoets2/bin/python")

# Load libraries
library(DESeq2)
library(tximport)
library(org.Mm.eg.db) # For gene ID to symbol mapping
library(AnnotationDbi)
library(Seurat)
library(SeuratDisk)
library(clusterProfiler)
library(dplyr)
library(tidyverse)
library(apeglm) # for lfc shrinkage
library(enrichplot)

library(ggplot2)
library(pheatmap)
library(grid)

library(DOSE)
library(ggrepel)

library(zellkonverter) # Alternative for reading .h5ad as SingleCellExperiment
library(reticulate)

library(Matrix)

library(CellChat)
library(patchwork)  # for combining plots

library(fgsea)
library(msigdbr)

library(plotly)
library(reshape2) # Load reshape2 package for melt

In [ ]:
path = '/data/bonn-epyc/projects/raw_data_huber/oliver_kretz/aging_kidney_bulk_IG25-IG48/'

# TxImport (Prepare data)

In [ ]:
# Import sample sheet w. sample names
sample_sheet = read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/Activin_kidney_info_selection.csv", header = TRUE, sep=",")
sample_sheet = dplyr::select(sample_sheet, 'sample_name', 'mutation', 'treatment1', 'treatment2', 'condition')

sample_sheet = sample_sheet %>% filter(sample_name %in% c("IG25", "IG26", "IG27", "IG37", "IG38", "IG39", "IG40", "IG41", "IG42", "IG43", "IG44", "IG45", "IG46", "IG47", "IG48"))
sample_sheet

In [ ]:
# Generate files input: sample vector with paths to quant.sf files
names = c(sample_sheet$sample_name)
samples= paste0(names, '/quant.sf')
samples= paste0('/data/bonn-epyc/projects/raw_data_huber/oliver_kretz/aging_kidney_bulk_aligned/star_salmon/', samples)

names(samples) = names
samples

In [ ]:
#Import tx2gene input: transcriptID, geneID
gene_map = read_tsv(file = '/data/bonn-epyc/projects/raw_data_huber/oliver_kretz/aging_kidney_bulk_aligned/star_salmon/salmon_tx2gene.tsv',  col_names=c( 'tranID1', 'geneID1', 'geneID2'), show_col_types = FALSE )
head(gene_map)
#gene_map = dplyr::select(gene_map, 'tranID', 'geneID')

In [ ]:
# Import count data
countData = tximport(files=samples, type= "salmon", tx2gene=gene_map , ignoreTxVersion = FALSE)
# Remove genes with NA entries
countData <- na.omit(countData)
#head(countData)

In [ ]:
sampleTable_full <- data.frame(condition = c(sample_sheet),
                          row.names = colnames(c(sample_sheet$sample_name)))

# Create DESeq()

In [ ]:
# generate dds object
dds = DESeqDataSetFromTximport(txi=countData,
                                            colData=as.data.frame(sample_sheet),
                                            design=~condition)

In [ ]:
# Normalize count data
dds <- estimateSizeFactors(dds)

# Pre-filter -> Keep only rows that have at least 10 reads
dim(dds)
keep = rowSums(counts(dds)) >= 10
dds = dds[keep,]
dim(dds)

# Fit the model
dds <- DESeq(dds)

# normalized dataset object
ntd <- normTransform(dds)

In [ ]:
resultsNames(dds)

# Calculate DEGs (with l2fc shrinkage)

In [ ]:
library(ashr)

# 1. Extract differential expression results
res_age_raw  <- results(dds, contrast = c("condition", "age", "ctrl"))
res_intv_raw <- results(dds, contrast = c("condition", "combi", "age"))
res_intv1_raw   <- results(dds, contrast = c("condition", "DR", "age"))
res_intv2_raw <- results(dds, contrast = c("condition", "sAct", "age"))
res_combi_ctrl_raw <- results(dds, contrast = c("condition", "combi", "ctrl"))

# Shrunken LFCs (PRESERVES YOUR CONTRAST DIRECTION)
res_age  <- lfcShrink(dds, contrast = c("condition", "age", "ctrl"), 
                      res = res_age_raw, type = "ashr")
res_intv <- lfcShrink(dds, contrast = c("condition", "combi", "age"), 
                      res = res_intv_raw, type = "ashr")
res_intv1    <- lfcShrink(dds, contrast = c("condition", "DR", "age"), 
                          res = res_intv1_raw, type = "ashr")
res_intv2    <- lfcShrink(dds, contrast = c("condition", "sAct", "age"), 
                          res = res_intv2_raw, type = "ashr")
res_combi_ctrl <- lfcShrink(dds, contrast = c("condition", "combi", "ctrl"), 
                            res = res_combi_ctrl_raw, type = "ashr")

In [ ]:
# 2. Filter significant DEGs changed by aging (padj < 0.05, |log2FC| > 1)
deg_age <- subset(as.data.frame(res_age), padj < 0.05 & abs(log2FoldChange) > 0)

# 3. Identify rescued genes: aging genes reversed by combined intervention
# Rescued if log2FC in intervention has opposite sign to aging
rescued_genes <- deg_age[which(sign(deg_age$log2FoldChange) != sign(res_intv[rownames(deg_age), 'log2FoldChange'])), ]

# 4. Add combined intervention vs ctrl results to rescued genes
rescued_genes$padj_combi_ctrl <- res_combi_ctrl[rownames(rescued_genes), 'padj']
rescued_genes$log2FC_combi_ctrl <- res_combi_ctrl[rownames(rescued_genes), 'log2FoldChange']

# 5. Filter to genes not significantly different from ctrl in combined intervention
#This means genes are not significantly different from control and have expression changes small enough to be considered close to normal.
rescued_to_normal <- rescued_genes[
  #(rescued_genes$padj_combi_ctrl > 0.05) & 
  (abs(rescued_genes$log2FC_combi_ctrl) < 0.5), ]

In [ ]:
#Calculate DEGs not rescued by any intervention
rescued_combi <- rownames(deg_age)[sign(deg_age$log2FoldChange) != sign(res_intv[rownames(deg_age), 'log2FoldChange'])]
rescued_DR    <- rownames(deg_age)[sign(deg_age$log2FoldChange) != sign(res_intv1[rownames(deg_age), 'log2FoldChange'])]
rescued_sAct  <- rownames(deg_age)[sign(deg_age$log2FoldChange) != sign(res_intv2[rownames(deg_age), 'log2FoldChange'])]

rescued_all <- unique(c(rescued_combi, rescued_DR, rescued_sAct))

non_rescued_genes <- setdiff(rownames(deg_age), rescued_all)
non_rescued_df <- deg_age[non_rescued_genes, ]

##  Map gene IDs to gene symbols 

In [ ]:
# 8. Map Ensembl IDs to gene symbols (already cleaned rownames)
add_gene_symbols <- function(df) {
  gene_symbols <- mapIds(org.Mm.eg.db,
                         keys = rownames(df),
                         column = 'SYMBOL',
                         keytype = 'ENSEMBL',
                         multiVals = 'first')
  df$Symbol <- gene_symbols
  return(df)
}

# Apply to each dataframe
rescued_genes <- add_gene_symbols(rescued_genes)
rescued_to_normal <- add_gene_symbols(rescued_to_normal)
deg_age <- add_gene_symbols(deg_age)
non_rescued_df <- add_gene_symbols(non_rescued_df)

# Intervention Effect Quantification 

In [ ]:
#### Compare which single intervention drove normalization most ####

In [ ]:
# Re-run or load differential results
#res_DR <- results(dds, contrast = c('condition', 'DR', 'age'))         # Aging + DR vs Aging
#res_sAct <- results(dds, contrast = c('condition', 'sAct', 'age'))     # Aging + sAct vs Aging

In [ ]:
# Subset for rescued genes going back to normal
reversal_genes <- rownames(rescued_to_normal)

In [ ]:
# Extract fold changes for all relevant comparisons
effect_comparison <- data.frame(
  Gene = reversal_genes,
  LFC_Age = res_age[reversal_genes, 'log2FoldChange'],
  LFC_Combined = res_intv[reversal_genes, 'log2FoldChange'],
  LFC_DR = res_intv1[reversal_genes, 'log2FoldChange'],
    LFC_sAct = res_intv2[reversal_genes, 'log2FoldChange']
)

# Make sure rescued_to_normal is indexed by rownames (gene IDs) and has a 'Symbol' column
# Assuming rescued_to_normal has rownames matching genes from effect_comparison$Gene and a column Symbol

# Create a named vector of symbols for fast lookup
symbol_lookup <- rescued_to_normal$Symbol
names(symbol_lookup) <- rownames(rescued_to_normal)

# Add a Symbol column next to the Gene column in effect_comparison
effect_comparison$Symbol <- symbol_lookup[effect_comparison$Gene]

# Optionally reorder columns to have Symbol after Gene
effect_comparison <- effect_comparison[, c("Gene", "Symbol", setdiff(colnames(effect_comparison), c("Gene", "Symbol")))]

head(effect_comparison)

In [ ]:
# Calculate which intervention restored closer to Combined intervention effect
# Use correlation or distance to evaluate similarity restoration pattern
# Only count restoration if signs oppose
effect_comparison$Restoration_DR <- ifelse(sign(effect_comparison$LFC_DR) != sign(effect_comparison$LFC_Age), 
                                          abs(effect_comparison$LFC_DR / effect_comparison$LFC_Age), 
                                          0)

effect_comparison$Restoration_sAct <- ifelse(sign(effect_comparison$LFC_sAct) != sign(effect_comparison$LFC_Age), 
                                            abs(effect_comparison$LFC_sAct / effect_comparison$LFC_Age), 
                                            0)

effect_comparison$Restoration_Combined <- ifelse(sign(effect_comparison$LFC_Combined) != sign(effect_comparison$LFC_Age), 
                                                abs(effect_comparison$LFC_Combined / effect_comparison$LFC_Age), 
                                                0)


# Quantify restoration strength
summary_stats <- data.frame(
  Mean_Restoration_DR = mean(effect_comparison$Restoration_DR, na.rm = TRUE),
  Mean_Restoration_sAct = mean(effect_comparison$Restoration_sAct, na.rm = TRUE),
  Mean_Restoration_Combined = mean(effect_comparison$Restoration_Combined, na.rm = TRUE)
)

summary_stats

In [ ]:
# add a column classifying whether the intervention impact on each gene is a “rescue” or “worsening” effect compared to aging
effect_comparison$Intervention_DR_Effect <- ifelse(
  sign(effect_comparison$LFC_DR) != sign(effect_comparison$LFC_Age),
  "Rescue",
  "Worsening"
)

effect_comparison$Intervention_sAct_Effect <- ifelse(
  sign(effect_comparison$LFC_sAct) != sign(effect_comparison$LFC_Age),
  "Rescue",
  "Worsening"
)

effect_comparison$Intervention_Combined_Effect <- ifelse(
  sign(effect_comparison$LFC_Combined) != sign(effect_comparison$LFC_Age),
  "Rescue",
  "Worsening"
)

head(effect_comparison)

## Split genes into those exclusively rescued by DR XOR sAct 

In [ ]:
#Split genes into those exclusively rescued by DR XOR sAct (restoration > 0 in only one)
exclusive_DR <- effect_comparison %>%
  filter((Restoration_DR > 0 & Restoration_sAct == 0))

exclusive_sAct <- effect_comparison %>%
  filter((Restoration_sAct > 0 & Restoration_DR == 0))

only_rescued_in_combi <- effect_comparison %>%
  filter((Restoration_sAct == 0 & Restoration_DR == 0))

head(exclusive_DR)
head(exclusive_sAct)
head(only_rescued_in_combi)

In [ ]:
length(effect_comparison$Gene)
length(rescued_to_normal$Symbol)
length(exclusive_DR$Gene)
length(exclusive_sAct$Gene)
length(only_rescued_in_combi$Gene)

length(deg_age$Symbol)

In [ ]:
deg_age

## Plots

In [ ]:
# Visualize restoration power per intervention
effect_long <- effect_comparison %>%
  tidyr::pivot_longer(cols = c(Restoration_DR, Restoration_sAct, Restoration_Combined),
                      names_to = "Intervention", values_to = "Restoration")
ggplot(effect_long, aes(x = Intervention, y = Restoration)) +
  geom_boxplot(fill = c("#F8766D", "#00BFC4", "#7CAE00")) +
  theme_minimal() +
  ylab("Restoration Ratio (vs aging)") +
  ggtitle("Comparative Restoration Efficiency of Interventions")

In [ ]:
# ── Output directory: re-run writes to a SEPARATE folder; old results_tables is left untouched ──
out_base <- "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk"
for (sub in c("", "intervention_impact", "enrichment", "celltype_markers", "celltype_pathways", "cellchat_outputs_kidney"))
  dir.create(file.path(out_base, sub), recursive = TRUE, showWarnings = FALSE)
cat("Outputs ->", out_base, "\n")


In [ ]:
# Step 6 (new): Compare proportions of gene categories
category_counts <- data.frame(
  Category = c("Exclusive_DR", "Exclusive_sAct", "only_rescued_in_combi", "Other"),
  Count = c(
    nrow(exclusive_DR),
    nrow(exclusive_sAct),
    nrow(only_rescued_in_combi),
    nrow(effect_comparison) - nrow(exclusive_DR) - nrow(exclusive_sAct) - nrow(only_rescued_in_combi)
  )
)

category_counts <- category_counts %>%
  mutate(Proportion = Count / sum(Count))

# Barplot of proportions
ggplot(category_counts, aes(x = Category, y = Proportion, fill = Category)) +
  geom_bar(stat = "identity") +
  geom_text(aes(label = scales::percent(Proportion, accuracy = 0.1)),
            vjust = -0.5, size = 3.5) +
  scale_fill_manual(values = c("Exclusive_DR" = "#F8766D",
                               "Exclusive_sAct" = "#00BFC4",
                               "only_rescued_in_combi" = "#7CAE00",
                               "Other" = "grey70")) +
  theme_minimal() +
  ylab("Proportion of Genes") +
  ggtitle("Proportion of Genes Exclusively Rescued by Each Intervention Set") +
  theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1))

# Optional: Save table for downstream pathway analyses
write.csv(category_counts, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_gene_category_proportions.csv", row.names = FALSE)

In [ ]:
# Convert proportions to percentage labels for the pie chart
category_counts <- category_counts %>%
  mutate(Percentage = scales::percent(Proportion, accuracy = 0.1))

# Pie chart with ggplot2
ggplot(category_counts, aes(x = "", y = Proportion, fill = Category)) +
  geom_col(width = 1, color = "white") +
  coord_polar(theta = "y") +
  geom_text(aes(label = Percentage), position = position_stack(vjust = 0.5)) +
  scale_fill_manual(values = c("Exclusive_DR" = "#F8766D",
                               "Exclusive_sAct" = "#00BFC4",
                               "only_rescued_in_combi" = "#7CAE00",
                               "Other" = "grey70")) +
  theme_void() +
  ggtitle("Proportion of Genes Exclusively Rescued by Each Intervention Set") +
  theme(plot.title = element_text(hjust = 0))


In [ ]:
top10_combined <- effect_comparison %>%
  arrange(desc(abs(Restoration_Combined))) %>%
  slice(1:10) %>%
 # select(Gene, Restoration_Combined) %>%
  rename(Restoration = Restoration_Combined) %>%
  mutate(Intervention = "Combined")

# Pivot longer to get data in plotting format
top10_long <- top10_combined %>%
  pivot_longer(
    cols = c(LFC_Age, LFC_DR, LFC_sAct, LFC_Combined),
    names_to = "Condition",
    values_to = "Log2FoldChange"
  ) %>%
  # Rename conditions for clarity
  mutate(Condition = recode(Condition,
                            LFC_Age = "Aging",
                            LFC_DR = "DR",
                            LFC_sAct = "sAct",
                            LFC_Combined = "Combined"))

# Plot
ggplot(top10_long, aes(x = reorder(Symbol, abs(Log2FoldChange)), y = Log2FoldChange, fill = Condition)) +
  geom_col(position = position_dodge(width = 0.8)) +
  coord_flip() +
  ylab("Log2 Fold Change") +
  xlab("Gene") +
  ggtitle("Top 10 Genes Restored by Combined Intervention & Impact of Single Interventions") +
  scale_fill_brewer(palette = "Set1") +
  theme_minimal() + theme(
    axis.text.y = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.4),
    plot.margin = margin(t = 15, r = 15, b = 5, l = 5)  # Increase right margin especially
  )

## Write out effect_comparison to csv

In [ ]:
# Write full effect_comparison
write.csv(effect_comparison, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/intervention_impact/kidney_intervention_impact_comparison.csv", row.names = TRUE)

#write exclusive_DR   & exclusive_sAct
write.csv(exclusive_DR, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/intervention_impact/kidney_intervention_impact_comparison_DR.csv", row.names = TRUE)
write.csv(exclusive_sAct, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/intervention_impact/kidney_intervention_impact_comparison_sAct.csv", row.names = TRUE)
# write out only rescued in combi genes
write.csv(only_rescued_in_combi, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/intervention_impact/kidney_intervention_impact_comparison_only_rescued_in_combi.csv", row.names = TRUE)

In [ ]:
only_rescued_in_combi

# Pathway Analysis

## GO BP

In [ ]:
# 7. Pathway enrichment for aging effect (Biological Process GO)
ego_age <- enrichGO(gene = rownames(deg_age),
                    OrgDb = org.Mm.eg.db,
                    keyType = 'ENSEMBL',
                    ont = 'BP',
                    pAdjustMethod = 'BH',
                    pvalueCutoff = 0.05,
                    qvalueCutoff = 0.2)

# 8. Pathway enrichment on rescued genes (Biological Process GO)
ego <- enrichGO(gene = rownames(rescued_genes),
                OrgDb = org.Mm.eg.db,
                keyType = 'ENSEMBL',
                ont = 'BP',
                pAdjustMethod = 'BH',
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.2)

# 8. Pathway enrichment on rescued-to-normal genes (Biological Process GO)
ego_norm <- enrichGO(gene = rownames(rescued_to_normal),
                OrgDb = org.Mm.eg.db,
                keyType = 'ENSEMBL',
                ont = 'BP',
                pAdjustMethod = 'BH',
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.2)

ego_norm_DR <- enrichGO(gene = exclusive_DR$Gene,
                OrgDb = org.Mm.eg.db,
                keyType = 'ENSEMBL',
                ont = 'BP',
                pAdjustMethod = 'BH',
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.2)

ego_norm_sAct <- enrichGO(gene = exclusive_sAct$Gene,
                OrgDb = org.Mm.eg.db,
                keyType = 'ENSEMBL',
                ont = 'BP',
                pAdjustMethod = 'BH',
                pvalueCutoff = 0.2,  # relaxed cutoff
                qvalueCutoff = 0.2)

# Cleaned Ensembl IDs for rescued_to_normal genes by combi that have no driving effect from sAct or DR alone
ego_norm_only_in_combi <- enrichGO(gene = only_rescued_in_combi$Gene,
                OrgDb = org.Mm.eg.db,
                keyType = 'ENSEMBL',
                ont = 'BP',
                pAdjustMethod = 'BH',
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.2)

ego_non_rescued <- enrichGO(gene = non_rescued_genes,
                OrgDb = org.Mm.eg.db,
                keyType = 'ENSEMBL',
                ont = 'BP',
                pAdjustMethod = 'BH',
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.2)

## GSEA

### For DEG Age & rescued

In [ ]:
# 1️⃣ Clean ENSEMBL IDs (remove version suffix after ".")
res_age$ENSEMBL_clean <- sub("\\..*$", "", rownames(res_age))
res_intv$ENSEMBL_clean <- sub("\\..*$", "", rownames(res_intv))
rescued_to_normal$ENSEMBL_clean <- sub("\\..*$", "", rownames(rescued_to_normal))

res_age_raw$ENSEMBL_clean <- sub("\\..*$", "", rownames(res_age_raw))
res_intv_raw$ENSEMBL_clean <- sub("\\..*$", "", rownames(res_intv_raw))

# 2️⃣ Map ENSEMBL IDs to gene symbols
res_age$Symbol <- mapIds(org.Mm.eg.db,
                         keys = res_age$ENSEMBL_clean,
                         column = "SYMBOL",
                         keytype = "ENSEMBL",
                         multiVals = "first")

res_age_raw$Symbol <- mapIds(org.Mm.eg.db,
                         keys = res_age_raw$ENSEMBL_clean,
                         column = "SYMBOL",
                         keytype = "ENSEMBL",
                         multiVals = "first")

res_intv$Symbol <- mapIds(org.Mm.eg.db,
                          keys = res_intv$ENSEMBL_clean,
                          column = "SYMBOL",
                          keytype = "ENSEMBL",
                          multiVals = "first")

res_intv_raw$Symbol <- mapIds(org.Mm.eg.db,
                          keys = res_intv_raw$ENSEMBL_clean,
                          column = "SYMBOL",
                          keytype = "ENSEMBL",
                          multiVals = "first")

rescued_to_normal$Symbol <- mapIds(org.Mm.eg.db,
                                   keys = rescued_to_normal$ENSEMBL_clean,
                                   column = "SYMBOL",
                                   keytype = "ENSEMBL",
                                   multiVals = "first")

# 3️⃣ Create ranked vectors for fgsea using the Wald stat
# 3️⃣ Create ranked vectors — avoid dplyr conflict with `stat`
ranked_age <- as.data.frame(res_age_raw)
ranked_age <- ranked_age[!is.na(ranked_age$stat) & !is.na(ranked_age$Symbol), ]
ranked_age <- ranked_age[order(-ranked_age$stat), ]
ranks_age <- setNames(ranked_age$stat, ranked_age$Symbol)

ranked_intv <- as.data.frame(res_intv_raw)
ranked_intv <- ranked_intv[!is.na(ranked_intv$stat) & !is.na(ranked_intv$Symbol), ]
ranked_intv <- ranked_intv[order(-ranked_intv$stat), ]
ranks_intv <- setNames(ranked_intv$stat, ranked_intv$Symbol)

# 4️⃣ Remove duplicate SYMBOLs
ranks_age <- ranks_age[!duplicated(names(ranks_age))]
ranks_intv <- ranks_intv[!duplicated(names(ranks_intv))]

# 4️⃣ Remove duplicate SYMBOLs
ranks_age <- ranks_age[!duplicated(names(ranks_age))]
ranks_intv <- ranks_intv[!duplicated(names(ranks_intv))]

# 5️⃣ Load mouse GO:BP pathways from MSigDB using current params
m_df <- msigdbr(species = "Mus musculus", collection = "C5", subcollection = "BP")
pathways <- split(m_df$gene_symbol, m_df$gs_name)

# 6️⃣ Run fgsea for aging and intervention datasets
fg_age <- fgsea(pathways = pathways, stats = ranks_age)
fg_intv <- fgsea(pathways = pathways, stats = ranks_intv)

# 6️⃣b NEW SECTION: Run fgsea for rescued genes using intervention stats
# 6b: GSEA for rescued genes using UNSHRUNKEN intervention stats
rescued_gene_ids <- rownames(rescued_to_normal)
resc_intv_stats <- res_intv_raw[rescued_gene_ids, "stat"]
# Get symbols from the shrunken object (which has them)
names(resc_intv_stats) <- res_intv[rescued_gene_ids, "Symbol"]
resc_intv_stats <- resc_intv_stats[!is.na(names(resc_intv_stats)) & !is.na(resc_intv_stats)]
resc_intv_stats <- resc_intv_stats[!duplicated(names(resc_intv_stats))]

fg_rescued <- fgsea(pathways = pathways,
                    stats = resc_intv_stats,
                    minSize = 15,
                    maxSize = 500)

# 7️⃣ Identify top significant pathways from aging
fg_age_sig <- fg_age %>%
  filter(padj < 0.05) %>%
  arrange(padj)
top_pathways <- fg_age_sig %>%
  head(30) %>%
  pull(pathway)

# 8️⃣ Build NES comparison matrix safely
fg_age_unique <- fg_age %>% filter(pathway %in% top_pathways) %>%
  distinct(pathway, .keep_all = TRUE) %>%
  select(pathway, NES_age = NES)

fg_intv_unique <- fg_intv %>% filter(pathway %in% top_pathways) %>%
  distinct(pathway, .keep_all = TRUE) %>%
  select(pathway, NES_intv = NES)

# DON'T filter fg_rescued by top_pathways; join later
fg_rescued_unique <- fg_rescued %>% distinct(pathway, .keep_all = TRUE) %>%
  select(pathway, NES_rescued = NES)

nes_mat <- fg_age_unique %>%
  left_join(fg_intv_unique, by = "pathway") %>%
  left_join(fg_rescued_unique, by = "pathway")

# 9️⃣ Inspect the NES matrix
head(nes_mat)

### For DEG exclusively rescued for DR and sAct

# Visualize bulk Results

## Generate boxplot for top rescued gene 

In [ ]:
# 3. Boxplot for specific gene expression in groups (example for the top rescued gene)
# Sort rescued_to_normal by padj_combi_ctrl
rescued_to_normal_sorted <- rescued_to_normal[order(rescued_to_normal$padj), ]

# Select top gene Ensembl ID and symbol
top_gene_id <- rownames(rescued_to_normal_sorted)[1]
top_gene_symbol <- rescued_to_normal_sorted$Symbol[1]

# Prepare data frame for plotting
df <- data.frame(
  Expression = assay(ntd)[top_gene_id, ],
  Condition = colData(dds)$condition
)

# Plot with gene symbol in the title
ggplot(df, aes(x=Condition, y=Expression, fill=Condition)) +
  geom_boxplot() +
  labs(title=paste("Expression of", top_gene_symbol), y="Normalized counts (log scale)") +
  theme_minimal()

In [ ]:
# PCA Plot with DESeq2 using vst-transformed counts
vsd <- vst(dds, blind=FALSE)
plotPCA(vsd, intgroup="condition") + ggtitle("PCA by condition")

## 1. DESeq2 results: MA-plot

In [ ]:
plotMA(res_age, main="MA-plot Aging vs Control", ylim=c(-5,5))
plotMA(res_intv, main="MA-plot Combined Intervention vs Aging", ylim=c(-5,5))

## 2. Volcano plot for aging DEGs

### Ctrl vs Age

In [ ]:
# Convert DESeqResults object to a data frame including gene names
res_age_df <- as.data.frame(res_age)

# Map gene IDs (ENSEMBL) in rownames of res_age_df to gene SYMBOLs
gene_symbols <- mapIds(org.Mm.eg.db,
                       keys = rownames(res_age_df),
                       column = "SYMBOL",
                       keytype = "ENSEMBL",
                       multiVals = "first")

# Add the gene symbols as a new column to the dataframe
res_age_df$Symbol <- gene_symbols

# If some genes didn't have a symbol mapping, you can keep their original IDs as fallback
res_age_df$Symbol[is.na(res_age_df$Symbol)] <- rownames(res_age_df)[is.na(res_age_df$Symbol)]


# Create a new variable for coloring by direction
res_age_df$significance <- "Not Significant"
res_age_df$significance[res_age_df$padj < 0.05 & res_age_df$log2FoldChange > 0] <- "Upregulated"
res_age_df$significance[res_age_df$padj < 0.05 & res_age_df$log2FoldChange < 0] <- "Downregulated"

# Select top genes to label by Symbol
top_genes <- res_age_df %>%
  filter(significance != "Not Significant") %>%
  arrange(padj) %>%
  slice_head(n = 15) %>%
  pull(Symbol)

ggplot(res_age_df, aes(x = log2FoldChange, y = -log10(padj), color = significance)) +
  geom_point(alpha = 0.6) +
  scale_color_manual(values = c("Upregulated" = "red", "Downregulated" = "blue", "Not Significant" = "grey")) +
  theme_minimal() +
  labs(title = "Volcano plot of Ctrl vs Aging DEGs") +
  geom_text_repel(data = subset(res_age_df, Symbol %in% top_genes),
                aes(label = Symbol), size = 3, max.overlaps = 10)

### Age vs Combi

In [ ]:
# Convert DESeqResults object to data frame including gene names
res_intv_df <- as.data.frame(res_intv)

# Map gene IDs (ENSEMBL) in rownames of res_intv_df to gene SYMBOLs
gene_symbols_intv <- mapIds(org.Mm.eg.db,
                            keys = rownames(res_intv_df),
                            column = "SYMBOL",
                            keytype = "ENSEMBL",
                            multiVals = "first")

# Add the gene symbols as a new column to the dataframe
res_intv_df$Symbol <- gene_symbols_intv

# If some genes didn't have a symbol mapping, keep original IDs as fallback
res_intv_df$Symbol[is.na(res_intv_df$Symbol)] <- rownames(res_intv_df)[is.na(res_intv_df$Symbol)]

# Create a new variable for coloring by direction with three categories
res_intv_df$significance <- "Not Significant"
res_intv_df$significance[res_intv_df$padj < 0.05 & res_intv_df$log2FoldChange > 0] <- "Upregulated"
res_intv_df$significance[res_intv_df$padj < 0.05 & res_intv_df$log2FoldChange < 0] <- "Downregulated"

# Select top genes to label (e.g., top 15 by smallest padj)
top_genes_intv <- res_intv_df %>%
  filter(significance != "Not Significant") %>%
  arrange(padj) %>%
  slice_head(n = 15) %>%
  pull(Symbol)


# Plot volcano plot with colored points and selective gene labels
ggplot(res_intv_df, aes(x = log2FoldChange, y = -log10(padj), color = significance)) +
  geom_point(alpha = 0.6) +
  scale_color_manual(values = c("Upregulated" = "red", "Downregulated" = "blue", "Not Significant" = "grey")) +
  theme_minimal() +
  labs(title = "Volcano plot of Aging vs Combination DEGs") +
  geom_text_repel(data = subset(res_intv_df, Symbol %in% top_genes_intv),
                  aes(label = Symbol), size = 3, max.overlaps = 10)

### Rescued by Combi

In [ ]:
# Convert DESeqResults to data frame and add gene symbols
res_age_df <- as.data.frame(res_age)
res_age_df$Symbol <- mapIds(org.Mm.eg.db,
                            keys=rownames(res_age_df),
                            column="SYMBOL",
                            keytype="ENSEMBL",
                            multiVals="first")
res_age_df$Symbol[is.na(res_age_df$Symbol)] <- rownames(res_age_df)[is.na(res_age_df$Symbol)]

# Define significant regulated genes (padj < 0.05 & |log2FC| > 1)
res_age_df$significant <- ifelse(res_age_df$padj < 0.05 & abs(res_age_df$log2FoldChange) > 0, "Significant", "Not Significant")

# Mark rescued genes among significant ones
res_age_df$highlight <- ifelse(rownames(res_age_df) %in% rownames(rescued_genes) & res_age_df$significant == "Significant",
                               "Rescued", "Other")

# Colors: rescued purple, others grey
colors <- c("Rescued" = "purple", "Other" = "grey")

# Select rescued genes for labeling (top 15 by padj)
top_rescued_genes <- res_age_df %>%
  filter(highlight == "Rescued") %>%
  arrange(padj) %>%
  slice_head(n = 15) %>%
  pull(Symbol)

ggplot(res_age_df, aes(x = log2FoldChange, y = -log10(padj), color = highlight)) +
  geom_point(alpha = 0.7) +
  scale_color_manual(values = colors) +
  theme_minimal() +
  labs(title = "Volcano plot with rescued genes highlighted") +
  geom_text_repel(data = subset(res_age_df, Symbol %in% top_rescued_genes),
                  aes(label = Symbol), size = 3, max.overlaps = 10)

#### Rescued by combi driven by SAct exclusively

In [ ]:
# Convert DESeqResults to data frame and add gene symbols
res_age_df <- as.data.frame(res_age)
res_age_df$Symbol <- mapIds(org.Mm.eg.db,
                            keys=rownames(res_age_df),
                            column="SYMBOL",
                            keytype="ENSEMBL",
                            multiVals="first")
res_age_df$Symbol[is.na(res_age_df$Symbol)] <- rownames(res_age_df)[is.na(res_age_df$Symbol)]

# Define significant regulated genes (padj < 0.05 & |log2FC| > 1)
res_age_df$significant <- ifelse(res_age_df$padj < 0.05 & abs(res_age_df$log2FoldChange) > 0, "Significant", "Not Significant")

# Mark rescued genes among significant ones
res_age_df$highlight <- ifelse(rownames(res_age_df) %in% exclusive_sAct$Gene & res_age_df$significant == "Significant",
                               "Rescued", "Other")

# Colors: rescued purple, others grey
colors <- c("Rescued" = "purple", "Other" = "grey")

# Select rescued genes for labeling (top 15 by padj)
top_rescued_genes <- res_age_df %>%
  filter(highlight == "Rescued") %>%
  arrange(padj) %>%
  slice_head(n = 15) %>%
  pull(Symbol)

ggplot(res_age_df, aes(x = log2FoldChange, y = -log10(padj), color = highlight)) +
  geom_point(alpha = 0.7) +
  scale_color_manual(values = colors) +
  theme_minimal() +
  labs(title = "Volcano plot with rescued genes driven by sAct highlighted") +
  geom_text_repel(data = subset(res_age_df, Symbol %in% top_rescued_genes),
                  aes(label = Symbol), size = 3, max.overlaps = 10)

#### Rescued by combi driven by DR exclusively

In [ ]:
# Convert DESeqResults to data frame and add gene symbols
res_age_df <- as.data.frame(res_age)
res_age_df$Symbol <- mapIds(org.Mm.eg.db,
                            keys=rownames(res_age_df),
                            column="SYMBOL",
                            keytype="ENSEMBL",
                            multiVals="first")
res_age_df$Symbol[is.na(res_age_df$Symbol)] <- rownames(res_age_df)[is.na(res_age_df$Symbol)]

# Define significant regulated genes (padj < 0.05 & |log2FC| > 1)
res_age_df$significant <- ifelse(res_age_df$padj < 0.05 & abs(res_age_df$log2FoldChange) > 0, "Significant", "Not Significant")

# Mark rescued genes among significant ones
res_age_df$highlight <- ifelse(rownames(res_age_df) %in% exclusive_DR$Gene & res_age_df$significant == "Significant",
                               "Rescued", "Other")

# Colors: rescued purple, others grey
colors <- c("Rescued" = "purple", "Other" = "grey")

# Select rescued genes for labeling (top 15 by padj)
top_rescued_genes <- res_age_df %>%
  filter(highlight == "Rescued") %>%
  arrange(padj) %>%
  slice_head(n = 15) %>%
  pull(Symbol)

ggplot(res_age_df, aes(x = log2FoldChange, y = -log10(padj), color = highlight)) +
  geom_point(alpha = 0.7) +
  scale_color_manual(values = colors) +
  theme_minimal() +
  labs(title = "Volcano plot with rescued genes driven by DR highlighted") +
  geom_text_repel(data = subset(res_age_df, Symbol %in% top_rescued_genes),
                  aes(label = Symbol), size = 3, max.overlaps = 10)

## 3. Heatmaps

### of rescued to normal genes expression

In [ ]:
# Filter samples to keep only "ctrl", "age", and "combi" conditions
keep_samples <- colData(dds)$condition %in% c("ctrl", "age", "combi")
rescued_counts <- counts(dds, normalized=TRUE)[rownames(rescued_to_normal), ]
rescued_counts_filtered <- rescued_counts[, keep_samples]

# Create annotation for the filtered samples
sample_anno <- as.data.frame(colData(dds)[keep_samples, "condition", drop=FALSE])

# Add condition labels to column names
colnames(rescued_counts_filtered) <- paste(colnames(rescued_counts_filtered), sample_anno$condition, sep = "_")

# Set rownames of annotation to be exactly matching column names of matrix
rownames(sample_anno) <- colnames(rescued_counts_filtered)

# Scale genes for visualization (row-wise scaling)
rescued_counts_scaled <- t(scale(t(rescued_counts_filtered)))

# Generate heatmap with filtered data and annotation
pheatmap(rescued_counts_scaled,
         annotation_col = sample_anno,
         main = "Expression heatmap of rescued genes returned to normal",
         fontsize_row = 6,
         cluster_cols = FALSE,
         show_rownames = FALSE)

In [ ]:
heatmap(rescued_counts_scaled, scale='row')

### of rescued genes expression

In [ ]:
# Filter colData and samples to keep only desired conditions
keep_samples <- colData(dds)$condition %in% c("ctrl", "age", "combi")
dds_filtered <- dds[, keep_samples]

# Make sure condition is a factor with defined levels
dds_filtered$condition <- factor(dds_filtered$condition, levels = c("ctrl", "age", "combi"))

# Re-estimate size factors and run DESeq
dds_filtered <- estimateSizeFactors(dds_filtered)
dds_filtered <- DESeq(dds_filtered)

# Variance stabilizing transformation on filtered dds
vsd_filtered <- vst(dds_filtered, blind = FALSE)

# Extract rescued gene counts for filtered samples (genes as rows)
rescued_gene_counts_filtered <- assay(vsd_filtered)[rownames(rescued_genes), ]

# Prepare combined names for columns (samples + condition)
combined_names <- paste(colnames(dds_filtered), dds_filtered$condition, sep = "_")

# Assign combined names to columns of expression matrix
colnames(rescued_gene_counts_filtered) <- combined_names

# Create annotation_col data frame with row names matching combined_names
annotation_col <- data.frame(condition = dds_filtered$condition)
rownames(annotation_col) <- combined_names

# Define annotation colors with exact matching levels
annotation_colors <- list(
  condition = c(ctrl = "skyblue", age = "orange", combi = "forestgreen")
)

# Plot heatmap with annotations
pheatmap(rescued_gene_counts_filtered,
         scale = "row",
         cluster_rows = TRUE,
         cluster_cols = TRUE,
         show_rownames = FALSE,
        # annotation_col = annotation_col,
        # annotation_colors = annotation_colors,
         main = "Heatmap of Rescued Genes (Filtered samples)")

In [ ]:
heatmap(rescued_gene_counts_filtered, scale='row')

### of aging genes expression

In [ ]:
# Generate heatmap data for Top 10 (assuming deg_age & dds are defined previously)
heatmap_data <- counts(dds)[rownames(deg_age), ]

# Modify row names to gene symbols
rownames(heatmap_data) <- deg_age$symbol

# Modify column names to include condition info
colnames(heatmap_data) <- paste(colnames(dds), dds$condition, sep = "-")

# Filter columns by conditions "ctrl", "age", and "combi"
conditions_to_keep <- c("ctrl", "age", "combi")
cols_to_keep <- grep(paste(conditions_to_keep, collapse="|"), colnames(heatmap_data))
heatmap_data_filtered <- heatmap_data[, cols_to_keep]

# Plot heatmap with adjusted options
options(repr.plot.width=7, repr.plot.height=10)

pheatmap(heatmap_data_filtered,
         #color = colorRampPalette(c("blue","white", "red"))(50),
         cluster_rows = TRUE, cluster_cols = TRUE,
         #show_rownames = TRUE, 
         show_rownames = FALSE, 
         scale = 'row', 
         main = "Heatmap of Aging Genes (Filtered samples)")

In [ ]:
heatmap(heatmap_data_filtered, scale='row')

## 4. GO enrichment dotplot

In [ ]:
?dotplot

In [ ]:
#Optional: Dotplot for pathway results
dotplot(ego_age, showCategory=20) + ggtitle("GO Biological Process enrichment of aging genes") + theme(plot.title = element_text(hjust = 0.5))
dotplot(ego, showCategory=20) + ggtitle("GO Biological Process enrichment of rescued genes") + theme(plot.title = element_text(hjust = 0.5))

#rescued to normal by combi
dotplot(ego_norm, showCategory=20, title=str_wrap("GO Biological Process enrichment of rescued-to-normal genes", width=40)) + theme(plot.title = element_text(hjust = 0.5))
dotplot(ego_norm_sAct, showCategory=20, title=str_wrap("GO Biological Process enrichment of rescued-to-normal driven by sAct genes", width=40)) + theme(plot.title = element_text(hjust = 0.5))
dotplot(ego_norm_DR, showCategory=20, title=str_wrap("GO Biological Process enrichment of rescued-to-normal driven by DR genes", width=40)) + theme(plot.title = element_text(hjust = 0.5))
dotplot(ego_norm_only_in_combi, showCategory=20, title=str_wrap("GO Biological Process enrichment of rescued-to-normal only in combi", width=40)) + theme(plot.title = element_text(hjust = 0.5))
dotplot(ego_non_rescued, showCategory=20) + ggtitle("GO Biological Process enrichment of genes not rescued by any intervention") + theme(plot.title = element_text(hjust = 0.5))

## 5. Barplot of top GO terms

In [ ]:
# Aging effect GO terms
barplot(ego_age, showCategory=10) + 
  ggtitle("GO Biological Process enrichment of aging genes") + 
  theme(plot.title = element_text(hjust = 0.5))

# Rescued gene GO terms
barplot(ego, showCategory=10) + 
  ggtitle("GO Biological Process enrichment of rescued genes") + 
  theme(plot.title = element_text(hjust = 0.5))

# Rescued to normal genes GO terms
barplot(ego_norm, showCategory=10) + 
  ggtitle("GO Biological Process enrichment of rescued-to-normal genes") + 
  theme(plot.title = element_text(hjust = 0.5))

barplot(ego_norm_sAct, showCategory=10) + 
  ggtitle("GO Biological Process enrichment of rescued-to-normal driven by sAct genes") + 
  theme(plot.title = element_text(hjust = 0.5))

barplot(ego_norm_DR, showCategory=10) + 
  ggtitle("GO Biological Process enrichment of rescued-to-normal driven by DR genes") + 
  theme(plot.title = element_text(hjust = 0.5))

barplot(ego_norm_only_in_combi, showCategory=10) + 
  ggtitle("GO Biological Process enrichment of rescued-to-normal only in combi") + 
  theme(plot.title = element_text(hjust = 0.5))

barplot(ego_non_rescued, showCategory=10) + 
  ggtitle("GO Biological Process enrichment of genes not rescued by any intervention") + 
  theme(plot.title = element_text(hjust = 0.5))

## 6. Plot GSEA Pathways

In [ ]:
# Set rownames and remove pathway column
#rownames(nes_mat) <- nes_mat$pathway
#nes_mat_plot <- nes_mat %>% select(-pathway)

# Make sure it’s a numeric matrix
nes_mat_plot <- as.matrix(nes_mat)
nes_mat_plot <- nes_mat %>%
  filter(!is.na(NES_age) & !is.na(NES_intv) & !is.na(NES_rescued)) %>%
  column_to_rownames(var = "pathway") %>%
  as.matrix()


options(repr.plot.width = 12, repr.plot.height = 20)  # inches
p <- pheatmap(nes_mat_plot,
              cluster_rows = TRUE,
              cluster_cols = FALSE,
              main = "Pathway NES, Kidney: Aging vs Combined Intervention",
              fontsize_row = 10,
              fontsize_col = 12,
              cellwidth = 25,
              cellheight = 15)

# Overall numbers plot

In [ ]:
library(ggplot2)
library(dplyr)

# Count total aging DEGs
n_deg_age <- nrow(deg_age)

# Count rescued genes (reversal upon combined intervention)
n_rescued <- nrow(rescued_genes)

# Count genes exclusively rescued by DR XOR sAct (restored > 0 in only one intervention)
nb_exclusive_DR <- effect_comparison %>%
  filter((Restoration_DR > 0 & Restoration_sAct == 0)) %>%
  nrow()

nb_exclusive_sAct <- effect_comparison %>%
  filter((Restoration_sAct > 0 & Restoration_DR == 0)) %>%
  nrow()

# Count genes rescued only in combined intervention (not rescued by DR or sAct alone)
only_combi <- effect_comparison %>%
  filter((Restoration_sAct == 0 & Restoration_DR == 0)) %>%
  nrow()

# Prepare summary data frame for plotting
summary_df <- tibble(
  Category = c("Aging DEGs", "Rescued Genes", "Exclusive DR Rescue", "Exclusive sAct Rescue", "Only Combined Rescue"),
  Count = c(n_deg_age, n_rescued, nb_exclusive_DR, nb_exclusive_sAct, only_combi)
)

# Plot bar chart
ggplot(summary_df, aes(x = Category, y = Count, fill = Category)) +
  geom_bar(stat = "identity") +
  geom_text(aes(label = Count), vjust = -0.5, size = 5) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Kidney Comparison of Differentially Expressed and Rescued Genes",
       y = "Number of Genes",
       x = "")


# Write out bulk DEG results

## DEG tables

In [ ]:
# Write full differential expression results with gene symbols and annotations
write.csv(res_age_df, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_res_age_results_with_symbols.csv", row.names = TRUE)
# Write out aged genes rescued by non of the interventions
write.csv(non_rescued_df, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_non_rescued_aged_DEGs.csv", row.names = TRUE)

In [ ]:
# Write rescued genes subset with gene symbols
# Convert res_intv results to a data frame for merging
res_intv_df <- as.data.frame(res_intv)

# Ensure gene IDs are rownames in both data frames
deg_age$GeneID <- rownames(deg_age)
res_intv_df$GeneID <- rownames(res_intv_df)

# Merge deg_age and res_intv_df by GeneID for rescued genes
combined_df <- merge(
  deg_age[, c("GeneID", "log2FoldChange", "padj")],  # aging DE stats
  res_intv_df[, c("GeneID", "log2FoldChange", "padj")],  # intervention DE stats
  by = "GeneID",
  suffixes = c("_aging", "_intervention")
)

# Optionally, filter combined_df to only rescued_genes if needed
rescued_gene_ids <- rownames(rescued_genes)
combined_rescued_df <- combined_df[combined_df$GeneID %in% rescued_gene_ids, ]

# Add gene symbols for clarity (optional)
gene_symbols <- mapIds(org.Mm.eg.db,
                       keys = combined_rescued_df$GeneID,
                       column = "SYMBOL",
                       keytype = "ENSEMBL",
                       multiVals = "first")
combined_rescued_df$Symbol <- gene_symbols
combined_rescued_df$LFC_Sum <- combined_rescued_df$log2FoldChange_aging + combined_rescued_df$log2FoldChange_intervention

# Add logical/rescue significance column based on padj_intervention threshold (e.g., 0.05)
combined_rescued_df$rescue_significant <- ifelse(combined_rescued_df$padj_intervention < 0.05, "Yes", "No")

# View or write combined table
head(combined_rescued_df)
write.csv(combined_rescued_df, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_combined_rescued_genes_DE_info.csv", row.names = FALSE)

# write  rescued_genes
write.csv(rescued_genes, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_rescued_genes.csv", row.names = TRUE)

# write rescued_gto_normal 
head(rescued_to_normal)
write.csv(rescued_to_normal, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_rescued_to_normal.csv", row.names = TRUE)

# Enrichment of significant Subset

In [ ]:
# ── Stage 2: significant rescued-to-normal (Stage 1 ∩ combi-vs-age padj < 0.05) ──
# Stage 1 = rescued_to_normal (built above with the SHRUNK res_combi_ctrl filter).
# Stage 2 additionally requires the combined-intervention effect (res_intv = combi vs age) to be significant.
rtn_ids  <- rownames(rescued_to_normal)
sig_mask <- !is.na(res_intv[rtn_ids, "padj"]) & res_intv[rtn_ids, "padj"] < 0.05
significant_rescued_to_normal <- rescued_to_normal[sig_mask, ]

cat("Stage 1 (rescued-to-normal):", nrow(rescued_to_normal), "\n")              # expect 2422
cat("Stage 2 (significant RTN):  ", nrow(significant_rescued_to_normal), "\n")   # expect 277

write.csv(significant_rescued_to_normal,
          "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_significant_rescued_to_normal.csv",
          row.names = TRUE)

# Stage-2 GO:BP enrichment (SAME settings as the other enrichGO calls above — no custom universe)
ego_sig <- enrichGO(gene = sub("\\..*$", "", rownames(significant_rescued_to_normal)),
                    OrgDb = org.Mm.eg.db, keyType = "ENSEMBL", ont = "BP",
                    pAdjustMethod = "BH", pvalueCutoff = 0.05, qvalueCutoff = 0.2)
write.csv(as.data.frame(ego_sig),
          "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/enrichment/kidney_enrichment_results_significant_rescued_to_normal.csv",
          row.names = FALSE)


## Enrichment

In [ ]:
# Export aging effect enrichment results to CSV
write.csv(as.data.frame(ego_age), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/enrichment/kidney_enrichment_results_aging.csv", row.names = FALSE, quote = FALSE)

# Export rescued genes enrichment results to CSV
write.csv(as.data.frame(ego), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/enrichment/kidney_enrichment_results_rescued.csv", row.names = FALSE, quote = FALSE)

# Export rescued-to-normal genes enrichment results to CSV
write.csv(as.data.frame(ego_norm), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/enrichment/kidney_enrichment_results_rescued-to-normal.csv", row.names = FALSE, quote = FALSE)
write.csv(as.data.frame(ego_norm_sAct), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/enrichment/kidney_enrichment_results_rescued-to-normal_sAct.csv", row.names = FALSE, quote = FALSE)
write.csv(as.data.frame(ego_norm_DR), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/enrichment/kidney_enrichment_results_rescued-to-normal__DR.csv", row.names = FALSE, quote = FALSE)
write.csv(as.data.frame(ego_norm_only_in_combi), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/enrichment/kidney_enrichment_results_rescued-to-normal_only_in_combi.csv", row.names = FALSE, quote = FALSE)

write.csv(as.data.frame(ego_non_rescued), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/enrichment/kidney_enrichment_results_non_rescued_genes.csv", row.names = FALSE, quote = FALSE)

In [ ]:
# Check Shrunken vs. Unshrunken restoration score

# Sn dataset

## --- LOAD SINGLE NUCLEI DATA FROM H5AD ---

In [ ]:
# 2. Import anndata and read .h5ad file
anndata <- import("anndata")
adata <- anndata$read_h5ad("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/annotated-aging-soupxed.h5ad")

# 3. Extract count matrix (transpose to genes x cells), gene names, and metadata
count_matrix <- t(as.matrix(adata$X))  
gene_names <- py_to_r(adata$var_names$tolist())
rownames(count_matrix) <- gene_names
cell_metadata <- as.data.frame(py_to_r(adata$obs))
rownames(cell_metadata) <- colnames(count_matrix)

# 4. Create Seurat object with count matrix and metadata
seurat_obj <- CreateSeuratObject(counts = count_matrix, meta.data = cell_metadata)

# 5. Add PCA and UMAP dimensional reductions if present in anndata
if (!is.null(adata$obsm)) {
  if ("X_pca" %in% names(adata$obsm)) {
    pca_mat <- py_to_r(adata$obsm[["X_pca"]])
    rownames(pca_mat) <- colnames(seurat_obj)
    colnames(pca_mat) <- paste0("PC_", 1:ncol(pca_mat))  # Fix column names warning
    seurat_obj[["pca"]] <- CreateDimReducObject(embeddings = pca_mat, key = "PC_")
  }
  
  if ("X_umap" %in% names(adata$obsm)) {
    umap_mat <- py_to_r(adata$obsm[["X_umap"]])
    rownames(umap_mat) <- colnames(seurat_obj)
    colnames(umap_mat) <- paste0("UMAP_", 1:ncol(umap_mat))  # Fix column names warning
    seurat_obj[["umap"]] <- CreateDimReducObject(embeddings = umap_mat, key = "UMAP_")
  }
}

# 6. Normalize data using LAYER SYNTAX (Seurat v5)
# Get counts layer
counts_layer <- LayerData(seurat_obj[["RNA"]], layer = "counts")

# Calculate total counts per cell
total_counts <- Matrix::colSums(counts_layer)
nonzero_cells <- total_counts > 0

# Normalize (CP10K + log1p)
norm_counts <- Matrix(0, nrow = nrow(counts_layer), ncol = ncol(counts_layer),
                      dimnames = dimnames(counts_layer))
norm_counts[, nonzero_cells] <- sweep(counts_layer[, nonzero_cells], 2, 
                                     total_counts[nonzero_cells], `/`) * 1e4

log_norm <- log1p(norm_counts)
log_norm[is.na(log_norm)] <- 0
log_norm[is.infinite(log_norm)] <- 0

# Store as "data" layer (Seurat v5 syntax)
LayerData(seurat_obj[["RNA"]], layer = "data") <- log_norm

print(Layers(seurat_obj[["RNA"]]))  # Confirm layers created

### Check Seurat Object

In [ ]:
# Check structure
head(rownames(seurat_obj))  # shows gene names used in Seurat object
head(seurat_obj@meta.data)

# If you have a column for cell types, for example "celltype"
if (!"celltype" %in% colnames(seurat_obj@meta.data)) {
  # If not, you need to assign or transfer cell type labels here before further analysis
  # For example, use cluster IDs as proxy:
  Idents(seurat_obj) <- seurat_obj$seurat_clusters
} else {
  # Use existing annotation
  Idents(seurat_obj) <- seurat_obj@meta.data$celltype
}

In [ ]:
# 1. Verify your Seurat object assay and features (genes)
DefaultAssay(seurat_obj)  # Should print "RNA" or the assay name you want to use
head(rownames(seurat_obj))  # confirm gene symbols here match those in genes_in_data
features_all <- rownames(seurat_obj)
cat("Number of genes in dataset:", length(features_all), "\n")

In [ ]:
# 2. Prepare your reversal gene list filtered to genes actually present in the Seurat object
reversal_genes <- na.omit(unique(rescued_to_normal$Symbol))
genes_in_data <- reversal_genes[reversal_genes %in% features_all]
cat("Number of reversal genes found in data:", length(genes_in_data), "\n")

if(length(genes_in_data) == 0){
  stop("No reversal genes match gene names in the Seurat object. Please check gene naming conventions.")
}

In [ ]:
# Your assay name is 'RNA'
# Check available layers in your assay
print(Layers(seurat_obj[["RNA"]]))  # Should show something or NULL
# Join layers for compatibility with functions like AddModuleScore
seurat_obj <- JoinLayers(seurat_obj, assay = "RNA", layers = Layers(seurat_obj[["RNA"]]))

# Check again layers after JoinLayers - should be just one now
print(Layers(seurat_obj[["RNA"]]))

In [ ]:
# Check available layers in RNA assay
print(Layers(seurat_obj[["RNA"]]))  # likely only "counts"

# Normalize counts and store as "data" layer in RNA assay
NormalizeData(seurat_obj, assay = "RNA", normalization.method = "LogNormalize", scale.factor = 1e4)

# Verify 'data' layer now exists
print(Layers(seurat_obj[["RNA"]]))  # Should show "counts" and "data"

### Filter for aged & rescued-to-normal (DR, sAct only and only_in_combi) genes

In [ ]:
# aged genes
genes_aged <- deg_age$Symbol
genes_aged <- genes_aged[genes_aged %in% rownames(seurat_obj)]
genes_aged <- unique(genes_aged)

# for reversal_genes
genes_norm <- reversal_genes
genes_norm <- genes_norm[genes_norm %in% rownames(seurat_obj)]
genes_norm <- unique(genes_norm)

# Use exclusive_DR and exclusive_sAct gene symbols from your effect_comparison table
genes_DR <- exclusive_DR$Symbol
genes_DR <- genes_DR[genes_DR %in% rownames(seurat_obj)]  # Ensure genes are in Seurat object
genes_DR <- unique(genes_DR)

genes_sAct <- exclusive_sAct$Symbol
genes_sAct <- genes_sAct[genes_sAct %in% rownames(seurat_obj)]
genes_sAct <- unique(genes_sAct)

# Add module scores to Seurat object
seurat_obj <- AddModuleScore(
  object = seurat_obj,
  features = list(genes_norm),
  name = "Reversal_norm"
)

seurat_obj <- AddModuleScore(
  object = seurat_obj,
  features = list(genes_DR),
  name = "Reversal_DR"
)

seurat_obj <- AddModuleScore(
  object = seurat_obj,
  features = list(genes_sAct),
  name = "Reversal_sAct"
)

seurat_obj <- AddModuleScore(
  object = seurat_obj,
  features = list(genes_aged),
  name = "Aged"
)

# Sort scores
# Function to reorder celltype by average expression of a feature
reorder_celltypes <- function(seurat_obj, feature) {
  avg_exp <- FetchData(seurat_obj, vars = c(feature, "celltype")) %>%
    group_by(celltype) %>%
    summarise(mean_exp = mean(get(feature), na.rm = TRUE)) %>%
    arrange(desc(mean_exp))
  factor_levels <- avg_exp$celltype
  seurat_obj$celltype <- factor(seurat_obj$celltype, levels = factor_levels)
  return(seurat_obj)
}

# For Reversal_norm1
seurat_obj <- reorder_celltypes(seurat_obj, "Reversal_norm1")
# To visualize, plot FeaturePlots or ViolinPlots by module score
VlnPlot(seurat_obj, features = "Reversal_norm1", group.by = "celltype", pt.size = 0) +
  ggtitle("Module Score: Reversal back-to-normal Genes") +    # Your custom title here
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.title.x = element_blank(),
    plot.margin = margin(t=10, r=30, b=10, l=10)
  )
# For Reversal_DR1
seurat_obj <- reorder_celltypes(seurat_obj, "Reversal_DR1")
# To visualize, plot FeaturePlots or ViolinPlots by module score
VlnPlot(seurat_obj, features = "Reversal_DR1", group.by = "celltype", pt.size = 0) +
  ggtitle("Module Score: Exclusive back-to-normal DR driven Genes") +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.title.x = element_blank(),
    plot.margin = margin(t=10, r=30, b=10, l=10)
  )

# For Reversal_sAct1
seurat_obj <- reorder_celltypes(seurat_obj, "Reversal_sAct1")
# To visualize, plot FeaturePlots or ViolinPlots by module score
VlnPlot(seurat_obj, features = "Reversal_sAct1", group.by = "celltype", pt.size = 0) +
  ggtitle("Module Score: Exclusive back-to-normal sAct driven Genes") +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.title.x = element_blank(),
    plot.margin = margin(t=10, r=30, b=10, l=10)
  )

# For Aged
seurat_obj <- reorder_celltypes(seurat_obj, "Aged1")
# To visualize, plot FeaturePlots or ViolinPlots by module score
VlnPlot(seurat_obj, features = "Aged1", group.by = "celltype", pt.size = 0) +
  ggtitle("Module Score: Aged Genes") +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 14),   # Bigger x-axis text
    axis.title.x = element_blank(),
    plot.title = element_text(size = 16),                          # Bigger title
    axis.text.y = element_text(size = 14),                          # Bigger y-axis text
    axis.title.y = element_text(size = 14),
    plot.margin = margin(t = 10, r = 30, b = 10, l = 10)
  )

In [ ]:
#Reversal Score Tables
group_col <- if ("celltype" %in% colnames(seurat_obj@meta.data)) "celltype" else "seurat_clusters"

# Combined reversal score table, sorted descending
avg_score_combined <- seurat_obj@meta.data %>%
  group_by(across(all_of(group_col))) %>%
  summarise(AverageReversalCombined = mean(Reversal_norm1, na.rm = TRUE)) %>%
  arrange(desc(AverageReversalCombined))

# DR reversal score table, sorted descending
avg_score_DR <- seurat_obj@meta.data %>%
  group_by(across(all_of(group_col))) %>%
  summarise(AverageReversalDR = mean(Reversal_DR1, na.rm = TRUE)) %>%
  arrange(desc(AverageReversalDR))

# sAct reversal score table, sorted descending
avg_score_sAct <- seurat_obj@meta.data %>%
  group_by(across(all_of(group_col))) %>%
  summarise(AverageReversalsAct = mean(Reversal_sAct1, na.rm = TRUE)) %>%
  arrange(desc(AverageReversalsAct))

# aged score table, sorted descending
avg_score_aged <- seurat_obj@meta.data %>%
  group_by(across(all_of(group_col))) %>%
  summarise(AverageAged = mean(Aged1, na.rm = TRUE)) %>%
  arrange(desc(AverageAged))

print(avg_score_combined)
print(avg_score_DR)
print(avg_score_sAct)
print(avg_score_aged)

In [ ]:
avg_score_aged <- seurat_obj@meta.data %>%
  group_by(across(all_of(group_col))) %>%
  summarise(AverageAged = mean(Aged1, na.rm = TRUE)) %>%
  arrange(desc(AverageAged))

head(avg_score_aged)

 ### Save or Export the Results for Downstream Analysis or Visualization

In [ ]:
# Extract metadata columns of interest - replace 'clusters' with your actual clustering column name
metadata_with_reversal <- seurat_obj@meta.data[, c("Reversal_norm1", "Reversal_DR1", "Reversal_sAct1", "celltype")]

# Write to CSV
write.csv(metadata_with_reversal, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_ReversalGeneScores_perCell_seurat.csv", row.names = TRUE)

In [ ]:
colnames(seurat_obj@meta.data)  # search for "aged" or "aging"

# Then add it:
metadata_aging <- seurat_obj@meta.data[, c("Aged1", "celltype")]
write.csv(metadata_aging, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_AgingGeneScores_perCell_seurat.csv", row.names = TRUE)

## Identify marker genes per cell type and filter for contributing aged and reversal genes

In [ ]:
Layers(seurat_obj[["RNA"]])

In [ ]:
# Set cell type identities
Idents(seurat_obj) <- seurat_obj$celltype  # or seurat_clusters

# 1. Find markers for all cell types/clusters
markers_all <- FindAllMarkers(seurat_obj, only.pos = TRUE, min.pct = 0.1, logfc.threshold = 0.25)

In [ ]:
# Prepare gene lists (cleaned, no NA)
reversal_genes_clean <- reversal_genes[!is.na(reversal_genes) & reversal_genes != ""]
aged_genes_clean <- deg_age$Symbol[!is.na(deg_age$Symbol) & deg_age$Symbol != ""]
aged_genes_only_rescued_in_combi_clean <- only_rescued_in_combi$Symbol[!is.na(only_rescued_in_combi$Symbol) & only_rescued_in_combi$Symbol != ""]

# 2. Subset markers for reversal genes, aged genes and thse genes only rescued with combination
markers_reversal <- markers_all[markers_all$gene %in% reversal_genes_clean, ]
markers_aged <- markers_all[markers_all$gene %in% aged_genes_clean, ]
markers_only_rescued_in_combi <- markers_all[markers_all$gene %in% aged_genes_only_rescued_in_combi_clean, ]

In [ ]:
dim(markers_all)

In [ ]:
dim(markers_aged)
dim(markers_reversal)
dim(markers_only_rescued_in_combi)

In [ ]:
# 3. Save full markers and subsets
write.csv(markers_all, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_markers/kidney_markers_all_celltypes.csv", row.names = FALSE)
write.csv(markers_reversal, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_markers/kidney_markers_reversal_genes_celltypes.csv", row.names = FALSE)
write.csv(markers_aged, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_markers/kidney_markers_aged_genes_celltypes.csv", row.names = FALSE)
write.csv(markers_only_rescued_in_combi, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_markers/kidney_markers_aged_genes_celltypes_only_rescued_in_combi.csv", row.names = FALSE)

# Replace '/' with '_' or other safe character in cluster names
# Function to sanitize cluster names to safe file names
sanitize_name <- function(x) {
  # Replace "/" with "_", you can add other replacements here if necessary
  gsub("/", "_", x)
}

# Use sanitized cluster names for writing files
safe_clusters_reversal <- sanitize_name(unique(markers_reversal$cluster))
for (ct in safe_clusters_reversal) {
  sub_df <- markers_reversal[sanitize_name(markers_reversal$cluster) == ct, ]
  write.csv(sub_df,
            paste0("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_markers/kidney_markers_reversal_", ct, ".csv"),
            row.names = FALSE)
}

safe_clusters_aged <- sanitize_name(unique(markers_aged$cluster))
for (ct in safe_clusters_aged) {
  sub_df <- markers_aged[sanitize_name(markers_aged$cluster) == ct, ]
  write.csv(sub_df,
            paste0("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_markers/kidney_markers_aged_", ct, ".csv"),
            row.names = FALSE)
}

# For markers_only_rescued_in_combi, make sure to get unique safely sanitized clusters
safe_clusters_only_rescued_in_combi <- sanitize_name(unique(markers_only_rescued_in_combi$cluster))
for (ct in safe_clusters_only_rescued_in_combi) {
  sub_df <- markers_only_rescued_in_combi[sanitize_name(markers_only_rescued_in_combi$cluster) == ct, ]
  write.csv(sub_df,
            paste0("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_markers/kidney_markers_only_rescued_in_combi_", ct, ".csv"),
            row.names = FALSE)
}

### Visualization % of genes per cell type 

In [ ]:
# Sanitize cluster names again if needed
markers_only_rescued_in_combi$cluster_safe <- gsub("/", "_", markers_only_rescued_in_combi$cluster)

# Count distinct genes per cluster
gene_count_table <- markers_only_rescued_in_combi %>%
  group_by(cluster_safe) %>%
  summarise(gene_count = n_distinct(gene))  # Replace 'gene' with your gene column name

# Print the table
print(gene_count_table)

# Optionally, write to CSV
write.csv(gene_count_table, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_markers/kidney_gene_counts_only_rescued_in_combi.csv",
          row.names = FALSE)

### GOBP enrichment per cell type (for aged and rescued markers)

In [ ]:
markers_aged <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_markers/kidney_markers_aged_genes_celltypes.csv")
markers_rescued <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_markers/kidney_markers_reversal_genes_celltypes.csv")

dim(markers_aged)
dim(markers_rescued)

In [ ]:
# Define your marker tables and analysis groups
marker_tables <- list(
  aged = markers_aged,
  rescued = markers_rescued
)

# Where to save enrichment tables and plots
table_dir <- "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/celltype_pathways/"
plot_dir <- file.path(table_dir, "enrichment_plots")

if (!dir.exists(table_dir)) dir.create(table_dir, recursive=TRUE)
if (!dir.exists(plot_dir)) dir.create(plot_dir, recursive=TRUE)

# Function to replace problematic characters in filenames
make_safe <- function(name) gsub("[/\\\\]", "_", name)

for (groupname in names(marker_tables)) {
  markers_df <- marker_tables[[groupname]]
  celltypes <- unique(markers_df$cluster)
  pathway_results <- list()
  
  # Run pathway analysis per cell type
  for (ct in celltypes) {
    genes <- markers_df$gene[markers_df$cluster == ct]
    entrez_ids <- na.omit(mapIds(org.Mm.eg.db, keys=genes, column="ENTREZID", keytype="SYMBOL", multiVals="first"))
    
    ego <- enrichGO(
      gene = entrez_ids,
      OrgDb = org.Mm.eg.db,
      keyType = "ENTREZID",
      ont = "BP",
      pAdjustMethod = "BH",
      pvalueCutoff = 0.05,
      qvalueCutoff = 0.2,
      readable = TRUE
    )
    pathway_results[[ct]] <- ego
    
    # Save enrichment table (as .csv)
    safe_ct <- make_safe(ct)
    if (!is.null(ego) & nrow(as.data.frame(ego)) > 0) {
      write.csv(as.data.frame(ego),
        file = file.path(table_dir, sprintf("gobp_%s_%s.csv", groupname, safe_ct)),
        row.names = FALSE
      )
    }
    
    # Plot and save barplot/dotplot
    if (!is.null(ego) & nrow(as.data.frame(ego)) > 0) {
      png(file = file.path(plot_dir, sprintf("GO_BP_barplot_%s_%s.png", groupname, safe_ct)), width=1000, height=800)
      print(barplot(ego, showCategory=10, title=paste("GO BP Enrichment -", groupname, ct)))
      dev.off()
      
      png(file = file.path(plot_dir, sprintf("GO_BP_dotplot_%s_%s.png", groupname, safe_ct)), width=1000, height=800)
      print(dotplot(ego, showCategory=10, title=paste("GO BP Enrichment -", groupname, ct)))
      dev.off()
    }
  }
}

# Cell–Cell Communication Analysis LR based (CellChat)

## use previously generated objects

In [ ]:
# Load your saved CellChat objects with centrality computed:
cellchat_sg18 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg18_centr.rds")
cellchat_sg20 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg20_centr.rds")
cellchat_sg24 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg24_centr.rds")
cellchat_sg26 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg26_centr.rds")
cellchat_sg28 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg28_centr.rds")

# reversal genes as symbols
reversal_genes <- na.omit(unique(rescued_to_normal$Symbol))

# 1) all LR interactions
all_communications_sg18 <- subsetCommunication(cellchat_sg18, slot.name = "net")  # ligand–receptor level[web:33][web:39]

# 2) filter by reversal genes
lr_filtered_sg18 <- all_communications_sg18 %>%
  filter(ligand %in% reversal_genes | receptor %in% reversal_genes)

cat("sg18: LR interactions involving reversal genes:", nrow(lr_filtered_sg18), "\n")
head(lr_filtered_sg18)

# 3) pathways containing reversal genes
all_pathways_sg18 <- cellchat_sg18@netP$pathways  # significant pathways[web:35][web:38]

pathway_has_reversal_genes <- function(pathway, obj, rev_genes) {
  pairs_df <- obj@LR$pairs[[pathway]]  # LR pairs used for this pathway[web:35][web:38]
  any(pairs_df$ligand %in% rev_genes) || any(pairs_df$receptor %in% rev_genes)
}

pathways_of_interest_sg18 <- Filter(function(p) pathway_has_reversal_genes(p, cellchat_sg18, reversal_genes),
                                    all_pathways_sg18)

cat("sg18: signaling pathways with reversal genes:\n")
print(pathways_of_interest_sg18)

# 4) visualize networks for those pathways
if (length(pathways_of_interest_sg18) > 0) {
  for (p in pathways_of_interest_sg18) {
    netVisual_circle(cellchat_sg18@netP$prob[,,p],    # pathway-level network[web:35][web:38]
                     vertex.weight = table(cellchat_sg18@idents),
                     weight.scale = TRUE,
                     title.name = paste("sg18:", p))
  }
} else {
  cat("sg18: no signaling pathways involving reversal genes found.\n")
}

# 5) write LR table
dir.create("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/cellchat_outputs_kidney", showWarnings = FALSE)
write.csv(lr_filtered_sg18,
          file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/cellchat_outputs_kidney/sg18_lr_interactions_reversal_genes.csv",
          row.names = FALSE)

In [ ]:
# Load CellChat object
cellchat_sg18 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg18_centr.rds")

# reversal genes as symbols
reversal_genes <- na.omit(unique(rescued_to_normal$Symbol))

# 1) all LR interactions (significant interactions detected)
all_communications_sg18 <- subsetCommunication(cellchat_sg18, slot.name = "net")

# 2) filter by reversal genes
lr_filtered_sg18 <- all_communications_sg18 %>%
  filter(ligand %in% reversal_genes | receptor %in% reversal_genes)

cat("sg18: LR interactions involving reversal genes:", nrow(lr_filtered_sg18), "\n")

# 3) Get pathways involving reversal genes from filtered LR interactions (no search in @LR$pairs)
pathways_of_interest_sg18 <- unique(lr_filtered_sg18$pathway_name)

cat("sg18: signaling pathways with reversal genes:\n")
print(pathways_of_interest_sg18)

# 4) visualize networks for those pathways using pathway-level probability
if (length(pathways_of_interest_sg18) > 0) {
  for (p in pathways_of_interest_sg18) {
    if (p %in% dimnames(cellchat_sg18@netP$prob)[[3]]) {
      netVisual_circle(cellchat_sg18@netP$prob[,,p], 
                       vertex.weight = table(cellchat_sg18@idents),
                       weight.scale = TRUE,
                       title.name = paste("sg18:", p))
    }
  }
} else {
  cat("sg18: no signaling pathways involving reversal genes found.\n")
}

# 5) write filtered LR table
dir.create("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/cellchat_outputs_kidney", showWarnings = FALSE)
write.csv(lr_filtered_sg18,
          file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/cellchat_outputs_kidney/sg18_lr_interactions_reversal_genes.csv",
          row.names = FALSE)


In [ ]:
pathways_of_interest_sg18 <- unique(lr_filtered_sg18$pathway_name)
pathways_of_interest_sg18

In [ ]:
# Load your saved CellChat objects with centrality computed
cellchat_sg18 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg18_centr.rds")
cellchat_sg20 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg20_centr.rds")
cellchat_sg24 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg24_centr.rds")
cellchat_sg26 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg26_centr.rds")
cellchat_sg28 <- readRDS("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/cellchat_objects/cellchat_sg28_centr.rds")

# reversal genes as symbols
reversal_genes <- na.omit(unique(rescued_to_normal$Symbol))

# Function to analyze and save circle plots and CSV for each sample
analyze_reversal_cellchat <- function(obj, name, rev_genes) {
  all_comm <- subsetCommunication(obj, slot.name = "net")
  lr_filt <- all_comm %>%
    filter(ligand %in% rev_genes | receptor %in% rev_genes)
  cat(name, ": LR with reversal genes:", nrow(lr_filt), "\n")
  
  # Extract pathways directly from filtered LR table column 'pathway_name'
  pathways_of_interest <- unique(lr_filt$pathway_name)
  cat(name, ": pathways with reversal genes:\n")
  print(pathways_of_interest)
  
  if (length(pathways_of_interest) > 0) {
    dir.create("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/cellchat_outputs_kidney", showWarnings = FALSE)
    for (p in pathways_of_interest) {
      if (p %in% dimnames(obj@netP$prob)[[3]]) {
        png(file = paste0("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/cellchat_outputs_kidney/", name, "_", gsub("[^A-Za-z0-9]", "_", p), ".png"),
            width = 800, height = 800, res = 100)
        netVisual_circle(obj@netP$prob[,,p],
                         vertex.weight = table(obj@idents),
                         weight.scale = TRUE,
                         title.name = paste(name, ":", p))
        dev.off()
        cat("Saved plot:", paste0(name, "_", p, ".png"), "\n")
      } else {
        cat("Pathway", p, "not found in netP probabilities for", name, "\n")
      }
    }
  } else {
    cat(name, ": no signaling pathways involving reversal genes found.\n")
  }
  
  write.csv(lr_filt,
            file = paste0("/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/cellchat_outputs_kidney/", name, "_lr_interactions_reversal_genes.csv"),
            row.names = FALSE)
}

# Create named list of your CellChat objects with new descriptive names
cellchat_list <- list(
  ctrl = cellchat_sg18,  # ctrl
  age = cellchat_sg20,   # age
  sAct = cellchat_sg24,  # sAct
  DR = cellchat_sg26,    # DR
  combi = cellchat_sg28  # combi
)

# Run analysis and save plots and CSVs for all samples in the renamed list
for (nm in names(cellchat_list)) {
  analyze_reversal_cellchat(cellchat_list[[nm]], nm, reversal_genes)
}

In [ ]:
head(reversal_genes)

## Visualization and Analysis

In [ ]:
if(length(pathways_of_interest) > 0) {
  for (p in pathways_of_interest) {
    netVisual_circle(cellchat_sg18@net[[p]]$count,
                     vertex.weight = table(cellchat_sg18@idents),
                     weight.scale = TRUE,
                     title.name = paste("Cell-cell communication network:", p))
  }
} else {
  message("No signaling pathways involving reversal genes found.")
}

# Explore CellChat summary plots generally or for specific pathways:

In [ ]:
netVisual_circle(cellchat_sg18@net$count, weight.scale = TRUE, label.edge = FALSE,
                 title.name = "Overall cell-cell interactions")

netAnalysis_signalingRole_network(cellchat_sg18)
netAnalysis_signalingRole_heatmap(cellchat_sg18)

netVisual_aggregate(cellchat_sg18, signaling = "ANGPT")  # or other pathway of interest

In [ ]:
# Visualize aggregate cell communication network
netVisual_circle(cellchat_sg18@net$count, weight.scale = TRUE, label.edge = FALSE,
                 title.name = "Number of interactions between cell types")

# Visualize major signaling inputs and outputs
netAnalysis_signalingRole_network(cellchat_sg18)

# Identify dominant senders and receivers
netAnalysis_signalingRole_heatmap(cellchat_sg18)

# Visualize specific signaling pathway, e.g. TGFb or VEGF
netVisual_aggregate(cellchat_sg18, signaling = "ANGPT")

# To focus on reversal gene-associated cell types, filter based on module scores from your single nucleus data
# For example, plot interactions involving cell types with highest 'Reversal' module score.

# Explore ligand-receptor pairs
pairLR <- subsetCommunication(cellchat_sg18)
head(pairLR)

In [ ]:
#  Optional Cross-Tissue Summary (Kidney vs Muscle)

In [ ]:
library(GSVA)
# Convert expression matrices to gene sets
bulk_expr <- assay(vsd)
gene_set <- list(Reversal = reversal_genes)
gsva_res <- gsva(bulk_expr, gene_set, method = "ssgsea")
heatmap(as.matrix(gsva_res), Rowv = NA, Colv = NA, scale = "none", col = viridis::viridis(50))

## (Optional) Comparative Analysis Across Experimental Conditions

In [ ]:
# Subset Seurat objects by condition
seurat_aging <- subset(seurat_obj, subset = condition == "age")
seurat_combined <- subset(seurat_obj, subset = condition == "combi")

cellchat_age <- createCellChat(seurat_aging, group.by = "celltype", assay = "RNA")
cellchat_combined <- createCellChat(seurat_combined, group.by = "celltype", assay = "RNA")

# Repeat preprocessing and inference steps for each
# Then compare using CellChat's comparative analysis functions
compare <- compareInteractions(cellchat_age, cellchat_combined)
netVisual_diffInteraction(compare)


# Check Gene Sets of interest

In [ ]:
# Define key genes of interest for kidney based on collaborator's input
kidney_genes <- c("Tgfb1", "Flt1", "B2m", "Gstm1", "Ccn4", "Prkcb", "S100a1", 
                  "Sp100", "Marcks", "Mndal", "Stk10", "Prkd1")

# Map gene symbols to Ensembl IDs if needed (assuming org.Mm.eg.db loaded)
kidney_ensembl <- mapIds(org.Mm.eg.db, keys = kidney_genes, column = "ENSEMBL", keytype = "SYMBOL", multiVals = "first")

# Subset DESeq results for kidney bulk data, e.g. rescued_to_normal or effect_comparison, focusing on these genes
kidney_subset <- effect_comparison[effect_comparison$Symbol %in% kidney_genes, ]

# View log2 fold changes by condition (age vs ctrl, DR, sAct, combined interventions)
print(kidney_subset[, c("Symbol", "LFC_Age", "LFC_DR", "LFC_sAct", "LFC_Combined")])

# Optional: generate plots for visualizing effect sizes per gene
library(ggplot2)
library(reshape2)
df_long <- melt(kidney_subset[, c("Symbol", "LFC_Age", "LFC_DR", "LFC_sAct", "LFC_Combined")], id.vars = "Symbol")
ggplot(df_long, aes(x = variable, y = value, fill = variable)) +
  geom_bar(stat="identity", position="dodge") +
  facet_wrap(~Symbol, scales = "free_y") +
  theme_bw() +
  labs(title = "Expression Changes of Key Kidney Genes Across Conditions", y = "Log2 Fold Change", x = "Condition")

In [ ]:
# Define key genes of interest for kidney based on collaborator's input
kidney_genes <- c("Flt1", "Jcad", "Fyn", "Id1", "Stab1", "Sp100", "Ramp2", "Cd34", "Nos3", "Cd40", "Emilin1")

# If rownames of res_age are Ensembl IDs with versions, clean them:
res_age$ENSEMBL_clean <- sub("\\..*$", "", rownames(res_age))
res_intv1$ENSEMBL_clean <- sub("\\..*$", "", rownames(res_intv1))
res_intv2$ENSEMBL_clean <- sub("\\..*$", "", rownames(res_intv2))
res_intv$ENSEMBL_clean <- sub("\\..*$", "", rownames(res_intv))

# Map Ensembl to Symbol for all DE results:
res_age$Symbol <- mapIds(org.Mm.eg.db, keys = res_age$ENSEMBL_clean, column = "SYMBOL", keytype = "ENSEMBL", multiVals = "first")
res_intv1$Symbol <- mapIds(org.Mm.eg.db, keys = res_intv1$ENSEMBL_clean, column = "SYMBOL", keytype = "ENSEMBL", multiVals = "first")
res_intv2$Symbol <- mapIds(org.Mm.eg.db, keys = res_intv2$ENSEMBL_clean, column = "SYMBOL", keytype = "ENSEMBL", multiVals = "first")
res_intv$Symbol <- mapIds(org.Mm.eg.db, keys = res_intv$ENSEMBL_clean, column = "SYMBOL", keytype = "ENSEMBL", multiVals = "first")

# Function to subset a results table by symbol vector
subset_by_symbols <- function(res_df, symbols) {
  subset(res_df, Symbol %in% symbols)
}

# Subset all results for your kidney genes
kidney_res_age <- subset_by_symbols(as.data.frame(res_age), kidney_genes)
kidney_res_intv1 <- subset_by_symbols(as.data.frame(res_intv1), kidney_genes)
kidney_res_intv2 <- subset_by_symbols(as.data.frame(res_intv2), kidney_genes)
kidney_res_intv <- subset_by_symbols(as.data.frame(res_intv), kidney_genes)

# Combine into one data frame for easier comparison
kidney_all_lfc <- data.frame(
  Symbol = kidney_genes
) %>%
  left_join(kidney_res_age[, c("Symbol", "log2FoldChange")], by = "Symbol") %>%
  rename(LFC_Age = log2FoldChange) %>%
  left_join(kidney_res_intv1[, c("Symbol", "log2FoldChange")], by = "Symbol") %>%
  rename(LFC_DR = log2FoldChange) %>%
  left_join(kidney_res_intv2[, c("Symbol", "log2FoldChange")], by = "Symbol") %>%
  rename(LFC_sAct = log2FoldChange) %>%
  left_join(kidney_res_intv[, c("Symbol", "log2FoldChange")], by = "Symbol") %>%
  rename(LFC_Combined = log2FoldChange)

print(kidney_all_lfc)

# Write out files for additional plots:

In [ ]:
# Export all module scores in one go
df <- data.frame(
  celltype        = seurat_obj$celltype,
  Reversal_norm   = seurat_obj@meta.data$Reversal_norm1,
  Reversal_DR     = seurat_obj@meta.data$Reversal_DR1,
  Reversal_sAct   = seurat_obj@meta.data$Reversal_sAct1,
  Aged            = seurat_obj@meta.data$Aged1
)
write.csv(df, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_module_scores_all.csv", row.names = FALSE)

cat("Exported", nrow(df), "cells,", length(unique(df$celltype)), "cell types\n")
cat("Columns:", colnames(df), "\n")

In [ ]:
# Check column names if unsure:
colnames(seurat_obj@meta.data)

# Kidney (after loading the kidney seurat_obj):
df <- data.frame(
  celltype  = seurat_obj$celltype,
  condition = seurat_obj$sample   # or whatever the condition column is called
)

write.csv(df, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_celltype_condition.csv", row.names = FALSE)

In [ ]:
# Kidney
# In R, using the anndata object that's already loaded:
umap_coords <- py_to_r(adata$obsm[["X_umap"]])
cell_metadata <- as.data.frame(py_to_r(adata$obs))

df <- data.frame(
  celltype  = cell_metadata$celltype,
  condition = cell_metadata$sample,   # or however condition is stored in the h5ad
  UMAP_1    = umap_coords[, 1],
  UMAP_2    = umap_coords[, 2]
)
write.csv(df, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/kidney_umap_coordinates.csv", row.names = FALSE)

In [ ]:
names(seurat_obj@reductions)

# Write Packages

In [ ]:
sessionInfo()

In [ ]:
pkgs <- installed.packages(fields = c("Package", "Version"))[, c("Package", "Version")]
write.csv(pkgs, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_kidney/results_tables_shrunk/installed_packages_kidney.csv", row.names = FALSE)

In [ ]:
installed.packages(fields = c("Package", "Version"))[, c("Package", "Version")]